# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a Croissant dataset using the `mlcroissant` library. We will walk through the process of data loading, overview, extraction, exploratory analysis, and visualization using the Kilifi County Mental Health Survey dataset.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (all referenced by `@id`).

In [ ]:
# Access all record sets in the metadata
record_sets = dataset._record_sets

print("Available Record Sets (@id and label):")
for rec in record_sets:
    print(f"  - @id: {rec['@id']} | name: {rec.get('name', '-')}")

# We'll use the first available record set for demonstration
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nMain record set selected for exploration: {main_record_set_id}")
    # Show its fields (columns)
    fields = rec.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields in this record set (@id and name):")
    for field in fields:
        print(f"  - Field @id: {field.get('@id', '?')} | name: {field.get('name', '-')}")
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s.

**Note:** We'll use the first record set found above.

In [ ]:
# Extract records from the main record set and show the DataFrame
record_sets_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Pick the main record set for further analysis
if dataframes:
    demo_recset_id = main_record_set_id
    print(f"\nColumns in record set {demo_recset_id}:")
    print(dataframes[demo_recset_id].columns.tolist())
    dataframes[demo_recset_id].head()
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by a key attribute.

We'll select a numeric field if available, and a grouping field such as demographic (e.g., 'age' or similar if present).

In [ ]:
# Identify candidate numeric and group fields
df = dataframes[demo_recset_id]

# Suggest likely numeric fields based on common mental health scores
possible_numeric_fields = [col for col in df.columns if any(k in col.lower() for k in ['gad', 'phq', 'psq', 'score', 'age'])]
numeric_field = None
if possible_numeric_fields:
    # We'll just take the first suitable match
    numeric_field = possible_numeric_fields[0]
else:
    # Default to first numeric column if available
    numeric_cols = df.select_dtypes(include='number').columns
    numeric_field = numeric_cols[0] if len(numeric_cols) else df.columns[0]
print(f"Using numeric field for analysis: {numeric_field}")

# Pick a grouping field (e.g., gender, education, etc.)
possible_group_fields = [col for col in df.columns if any(k in col.lower() for k in ['gender', 'sex', 'education', 'marital'])]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]
print(f"Using group field: {group_field}")

# Filter records where the numeric field > threshold (choose meaningful default)
try:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
except Exception:
    threshold = 10
filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df.copy()

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"{numeric_field} is not numeric and cannot be normalized.")

# Group by a demographic field and compute group means
if group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and show group means by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group if numeric
if group_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f'{numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load a Croissant dataset from a schema URL
- Explore available record sets and their fields (using `@id` references)
- Extract survey data into pandas DataFrames
- Apply simple filtering, normalization, and group-wise aggregation
- Visualize data distributions and demographic group differences

This demonstrates how to access and work with data using the standardized Croissant metadata approach, referencing all fields and record sets by their `@id` throughout.